[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jairomelo/aiOCR/blob/main/models/Florence-2-large/Florence-2-large.ipynb)

## Prerequisites

Runtime: Python 3, T4 GPU

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Florence-2 requires timm (vision backbone) and einops.
# Pin tokenizers<0.21: Florence-2's custom TokenizersBackend uses an API
# removed in tokenizers 0.21+ ("additional_special_tokens" attribute error).
%pip install -q "transformers>=4.41.0" "tokenizers<0.21" timm einops Pillow

In [3]:
from transformers import AutoProcessor, AutoModelForCausalLM

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

In [4]:
import torch

In [5]:
from pathlib import Path
from PIL import Image

WORKING_DIR = Path('/content/drive/MyDrive/aiOCR')
MODEL_NAME = 'microsoft/Florence-2-large'

# Florence-2-large: 0.77B params → ~1.5 GB fp16, well within T4's 15 GB.
# No quantization needed.
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    trust_remote_code=True,
).eval().cuda()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

modeling_florence2.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/microsoft/Florence-2-large:
- modeling_florence2.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

In [13]:
IMAGE_FILES = [
    WORKING_DIR / 'images/CCundinamarca/CCundinamarca_page_1.png',
    WORKING_DIR / 'images/CCundinamarca/CCundinamarca_page_46.png',
    WORKING_DIR / 'images/CO_18180627/CO_18180627_page_1.png',
    WORKING_DIR / 'images/dmcz_18250101/dmcz_18250101_page_1.png',
    WORKING_DIR / 'images/dmcz_18250101/dmcz_18250101_page_4.png',
    WORKING_DIR / 'images/el-redactor-1/el-redactor-1_page_1.png',
    WORKING_DIR / 'images/el-redactor-1/el-redactor-1_page_2.png',
    WORKING_DIR / 'images/pineda1/pineda1_page_1.png',
    WORKING_DIR / 'images/pineda1/pineda1_page_3.png',
    WORKING_DIR / 'images/pineda1/pineda1_page_4.png',
    WORKING_DIR / 'images/AR_SR8V4R3/AR_SR8V4R3_4.jpg',
]

## Inference

In [14]:
import time

transcription_out = WORKING_DIR / 'transcriptions/Florence-2-large'
transcription_out.mkdir(parents=True, exist_ok=True)

task_prompt = '<OCR>'

for IMAGE_FILE in IMAGE_FILES:
    image_stem = IMAGE_FILE.stem
    out_path = transcription_out / f'{image_stem}.md'

    if out_path.exists():
        print(f'Skipping (already done): {image_stem}')
        continue

    if not IMAGE_FILE.exists():
        print(f'Skipping (image not found): {IMAGE_FILE}')
        continue

    image = Image.open(IMAGE_FILE).convert('RGB')
    inputs = processor(
        text=task_prompt, images=image, return_tensors='pt'
    ).to('cuda', torch.float16)

    t0 = time.time()
    with torch.no_grad():
        generated_ids = model.generate(
            input_ids=inputs['input_ids'],
            pixel_values=inputs['pixel_values'],
            max_new_tokens=4096,
            do_sample=False,
            num_beams=3,
        )
    elapsed = time.time() - t0

    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
    parsed = processor.post_process_generation(
        generated_text,
        task=task_prompt,
        image_size=(image.width, image.height),
    )
    transcription = parsed[task_prompt]

    out_path.write_text(transcription, encoding='utf-8')
    print(f'Done in {elapsed:.1f}s — saved: transcriptions/Florence-2-large/{image_stem}.md')

Done in 11.3s
puede aguardar la sentencia que ha de entregarle incontente.
Escipente en el rostro, la abobetean, le azotam con varas
hasta dejar descubertas las venas i los huesos: el cuerpo de
la victima no mas que una liga de los pies a la cabeza.
A A la cruelda se junta una mofa insulante. Como el
tigre que juega con una presa antes de devoratante. Asi
aquel pueblo barbara ultraña una manso cordero antés de
verter su sangre. Le visiten una túnica de escarnio: le
ponen en la mano españa con a guisa de cetero en la cabeza
una corona de españas en senal de diadentes luego ven-
dándole los ojos doblan a rodilla, le dan estro en la bodefadas
en el estro de la bienención de la guerrero de los judios.
En la estro que el bieneno se público de los nacioni
Entre aquele juebo de verdado y la hallarría de la
poderosa bondo de el en persona en el de los suyos.
Purifice a los lepresas, restutuyo la vista a los gegos de el
oido y los sordos, libro y los endemonidos. Resucitid los
murfueros, a los 

### Saving the output